In [1]:
#Скачиваем все нужное
!pip install lxml pandas
!pip install nltk
!pip install requests
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 53.4 MB/s eta 0:00:00


In [2]:
#Скачиваем корпус с Гитхаба
import requests

url = 'https://github.com/UniversalDependencies/UD_Russian-GSD/blob/master/ru_gsd-ud-dev.conllu'
file_name = 'UD_GSD.txt'

response = requests.get(url)

if response.status_code == 200:
    with open('UD_GSD.txt', 'wb') as file:
        file.write(response.content)
    print(f'Файл успешно скачан.')
else:
    print(f'Ошибка при скачивании файла')

Файл успешно скачан.


In [3]:
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import pymorphy3
import re

nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [25]:
with open('UD_GSD.txt', 'r', encoding='utf-8') as file:
    text = file.read()

morph = pymorphy3.MorphAnalyzer()

lemma = input("Введите лемму: ")

normalized_lemma = morph.normal_forms(lemma)[0]

sentences = sent_tokenize(text)

examples = []
for sentence in sentences:
    tokens = nltk.word_tokenize(sentence)
    lemmas = [morph.normal_forms(token)[0] for token in tokens]
    poses = [morph.POS(token)[0] for token in tokens]
    if normalized_lemma in lemmas:
        examples.append(sentence)
        if len(examples) == 5:  # Выводим не более 5 примеров
            break

# Выводим примеры употребления леммы
if examples:
    print(f"\nПримеры употребления леммы '{lemma}':")
    for i, example in enumerate(examples, 1):
        match = re.search('text =([^\n]*)', example)
        #print(match.group(1))
        print(f"{i}. {match.group(1)}")
else:
    print(f"\nЛемма '{lemma}' не найдена в корпусе.")

Введите лемму: университет

Примеры употребления леммы 'университет':
1.  Из Томска он переехал в Москву, где поступил в Московский университет на историко-филологический факультет, курс которого окончил в 1890 году с дипломом I степени.
2.  К заседанию G8 под предводительством Великобритании в Белфасте Джеффри Оуэнс (Jeffrey Owens) бывший глава налогового подразделения ОЭСР и Мик Мур (Mick Moore) из Университета Сассекса (the University of Sussex) представили свой план действий для стран, участниц ``большой восьмерки\u0026#39;\u0026#39;, который должен стать значительным дополнением для повестки дня заседания G20 в Санкт-Петербурге (Россия) в сентябре 2013 года.
3.  Университе́т Аделаи́ды -- государственный университет Австралии, один из старейших в стране, основанный в 1874 году.
4.  Белый цвет (серебро) -- символ чистоты, совершенства, мира и взаимопонимания","1\tБелый\tбелый\tADJ\tJJL\tCase=Nom|Degree=Pos|Gender=Masc|Number=Sing\t2\tamod\t_\t_","2\tцвет\tцвет\tNOUN\tNN\tAnimacy=Ina

Мы столкнулись с такой проблемой, что выводил сначала теги предложения, а затем само предложение. Мы попытались решить эту проблему через регулярные выражения:

In [20]:
# Поиск всего после 'text', включая символы после него
import re

text = '1. ","1\tВ\tв\tADP\tIN\t_\t3\tcase\t_\t_","2\tэтом\tэтот\tDET\tDT\tCase=Loc|Gender=Masc|Number=Sing\t3\tdet\t_\t_","3\tгоду\tгод\tNOUN\tNN\tAnimacy=Inan|Case=Loc|Gender=Masc|Number=Sing\t9\tobl\t_\t_","4\tСаймон\tСаймон\tPROPN\tNNP\tAnimacy=Anim|Case=Nom|Gender=Masc|Number=Sing\t9\tnsubj\t_\t_","5\tМактэвиш\tМактэвиш\tPROPN\tNNP\tAnimacy=Anim|Case=Nom|Gender=Masc|Number=Sing\t4\tflat:name\t_\t_","6\tи\tи\tCCONJ\tCC\t_\t7\tcc\t_\t_","7\tДжон\tДжон\tPROPN\tNNP\tAnimacy=Anim|Case=Nom|Gender=Masc|Number=Sing\t4\tconj\t_\t_","8\tФрейзер\tФрейзер\tPROPN\tNNP\tAnimacy=Anim|Case=Nom|Gender=Masc|Number=Sing\t7\tflat:name\t_\t_","9\tосновали\tосновать\tVERB\tVBC\tAspect=Perf|Mood=Ind|Number=Plur|Tense=Past|VerbForm=Fin|Voice=Act\t0\troot\t_\t_","10\tкомпанию\tкомпания\tNOUN\tNN\tAnimacy=Inan|Case=Acc|Gender=Fem|Number=Sing\t9\tobj\t_\t_","11\tв\tв\tADP\tIN\t_\t12\tcase\t_\t_","12\tЛондоне\tЛондон\tPROPN\tNNP\tAnimacy=Inan|Case=Loc|Gender=Masc|Number=Sing\t10\tnmod\t_\tSpaceAfter=No","13\t,\t,\tPUNCT\t,\t_\t10\tpunct\t_\t_","14\tMcTavish\tMctavish\tX\tFW\tForeign=Yes\t10\tappos\t_\tSpaceAfter=No","15\t,\t,\tPUNCT\t,\t_\t16\tpunct\t_\t_","16\tFraser\tFraser\tX\tFW\tForeign=Yes\t14\tconj\t_\t_","17\tand\tand\tX\tFW\tForeign=Yes\t18\tcc\t_\t_","18\tCompany\tcompany\tX\tFW\tForeign=Yes\t14\tconj\t_\tSpaceAfter=No","19\t,\t,\tPUNCT\t,\t_\t14\tpunct\t_\t_","20\tзадачей\tзадача\tNOUN\tNN\tAnimacy=Inan|Case=Ins|Gender=Fem|Number=Sing\t10\tacl:relcl\t_\t_","21\tкоторой\tкоторый\tPRON\tAWP\tAnimacy=Inan|Case=Gen|Gender=Fem|Number=Sing\t20\tnmod\t_\t_","22\tбыло\tбыть\tAUX\tVBC\tAspect=Imp|Gender=Neut|Mood=Ind|Number=Sing|Tense=Past|VerbForm=Fin\t20\tcop\t_\t_","23\tснабжение\tснабжение\tNOUN\tNN\tAnimacy=Inan|Case=Nom|Gender=Neut|Number=Sing\t20\tnsubj\t_\t_","24\tСеверо-Западной\tСеверо-западный\tADJ\tJJL\tCase=Gen|Degree=Pos|Gender=Fem|Number=Sing\t25\tamod\t_\t_","25\tкомпании\tкомпания\tNOUN\tNN\tAnimacy=Inan|Case=Gen|Gender=Fem|Number=Sing\t23\tnmod\t_\t_","26\tпродуктами\tпродукт\tNOUN\tNN\tAnimacy=Inan|Case=Ins|Gender=Masc|Number=Plur\t23\tnmod\t_\t_","27\tи\tи\tCCONJ\tCC\t_\t28\tcc\t_\t_","28\tсбыт\tсбыт\tNOUN\tNN\tAnimacy=Inan|Case=Nom|Gender=Masc|Number=Sing\t23\tconj\t_\t_","29\tмеха\tмех\tNOUN\tNN\tAnimacy=Inan|Case=Gen|Gender=Masc|Number=Sing\t28\tnmod\t_\tSpaceAfter=No","30\t.\t.\tPUNCT\t.\t_\t9\tpunct\t_\t_","","# sent_id = dev-s103","# text = Из Томска он переехал в Москву, где поступил в Московский университет на историко-филологический факультет, курс которого окончил в 1890 году с дипломом I степени.'


match = re.search('text =([^\n]*)', text)
if match:
    print("Всё после 'text':", match.group(1))
else:
    print("Слово 'text' не найдено")


Всё после 'text':  Из Томска он переехал в Москву, где поступил в Московский университет на историко-филологический факультет, курс которого окончил в 1890 году с дипломом I степени.
